# Automobile Segmentation with Hierarchical Clustering

**Project type:** Unsupervised machine learning  
**Dataset:** `mtcars.csv` (32 automobile models, 11 numeric features)  
**Tools:** Python, pandas, NumPy, Matplotlib, Seaborn, SciPy, scikit-learn

## Business / analytical question

Can automobile models be grouped into interpretable segments based on fuel economy, engine, transmission, gearing, and vehicle characteristics?

This is an exploratory segmentation exercise. The dataset is small and historical, so the results should not be generalized to the entire automobile market.

## 1. Analytical workflow

1. Load and inspect the data.
2. Check data quality and understand feature distributions.
3. Standardize numeric features because hierarchical clustering uses distances.
4. Compare Average and Ward linkage.
5. Evaluate candidate cluster counts using silhouette scores and cluster sizes.
6. Select a practical solution, profile the groups, and visualize them with PCA.
7. State limitations and recommendations.

**Important:** Cluster count selection is not based on silhouette score alone. Interpretability, cluster sizes, feature choices, and the analytical purpose also matter.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

from scipy.cluster.hierarchy import linkage, dendrogram, fcluster

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

# If running in Google Colab, upload mtcars.csv or mount Google Drive.
# Keep mtcars.csv in the same folder as this notebook for the simplest setup.
automobile = pd.read_csv("mtcars.csv")

automobile.head()

## 2. Data quality and structure

The dataset contains one row per automobile model. The `model` column is an identifier and should not be used as a numeric clustering feature.

The checks below review dimensions, data types, missing values, duplicate records, and descriptive statistics.

In [ ]:
print("Shape:", automobile.shape)
print("\nData types:")
display(automobile.dtypes.to_frame("dtype"))

print("\nMissing values:")
display(automobile.isna().sum().to_frame("missing_values"))

print("\nDuplicate rows:", automobile.duplicated().sum())

print("\nDescriptive statistics:")
display(automobile.describe().T.round(2))

## 3. Feature selection and scaling

The analysis uses the 11 numeric variables in the dataset. This includes `vs` (engine configuration) and `am` (transmission), which are binary-coded variables.

Standardization prevents variables with larger measurement scales from automatically dominating Euclidean distances. However, standardizing binary indicators gives them a particular influence in the distance calculation. Treat this as a modeling choice and consider sensitivity testing with alternative feature sets or weights.

In [ ]:
feature_cols = [col for col in automobile.columns if col != "model"]
X = automobile[feature_cols].copy()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Features used:", feature_cols)
print("Scaled matrix shape:", X_scaled.shape)

## 4. Hierarchical clustering and dendrograms

Average linkage uses the average pairwise distance between observations in two clusters. Ward linkage merges clusters to minimize the increase in within-cluster variance; it is commonly used with Euclidean distance.

The dendrograms help inspect the hierarchy, but a visual cut should be supported by quantitative evaluation and cluster interpretation.

In [ ]:
Z_average = linkage(X_scaled, method="average", metric="euclidean")
Z_ward = linkage(X_scaled, method="ward", metric="euclidean")

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

dendrogram(
    Z_average, labels=automobile["model"].tolist(),
    leaf_rotation=90, ax=axes[0]
)
axes[0].set_title("Average Linkage Dendrogram")
axes[0].set_ylabel("Euclidean distance")

dendrogram(
    Z_ward, labels=automobile["model"].tolist(),
    leaf_rotation=90, ax=axes[1]
)
axes[1].set_title("Ward Linkage Dendrogram")
axes[1].set_ylabel("Ward distance")

plt.tight_layout()
plt.show()

## 5. Compare candidate cluster counts

The silhouette score measures how close observations are to their own cluster compared with other clusters. Higher values generally indicate better separation, but a high score does not automatically mean a solution is useful.

We compare two linkage methods across 2–7 clusters and report cluster sizes alongside the score. This makes tiny or highly imbalanced clusters visible.

In [ ]:
comparison_rows = []

for method, Z in [("Average", Z_average), ("Ward", Z_ward)]:
    for k in range(2, 8):
        labels = fcluster(Z, t=k, criterion="maxclust")
        sizes = pd.Series(labels).value_counts().sort_index().tolist()

        comparison_rows.append({
            "linkage": method,
            "requested_clusters": k,
            "actual_clusters": len(np.unique(labels)),
            "silhouette": silhouette_score(X_scaled, labels),
            "cluster_sizes": sizes
        })

comparison = pd.DataFrame(comparison_rows)
display(comparison.round({"silhouette": 3}))

### Selection rationale

For the worked example below, we use **Ward linkage with three clusters** as a practical, interpretable segmentation with cluster sizes of 15, 12, and 5 observations.

This is not the unique mathematical optimum: some four- and five-cluster solutions have slightly higher silhouette scores. The three-cluster solution is chosen as a balance between separation, group size, and ease of communication. If the business objective changes, the selected number of clusters may change.

In [ ]:
selected_linkage = "ward"
selected_k = 3

selected_Z = Z_ward if selected_linkage == "ward" else Z_average
cluster_labels = fcluster(selected_Z, t=selected_k, criterion="maxclust")

automobile_clustered = automobile.copy()
automobile_clustered["Cluster"] = cluster_labels

print("Linkage:", selected_linkage)
print("Number of clusters:", automobile_clustered["Cluster"].nunique())
print("Silhouette score:", round(silhouette_score(X_scaled, cluster_labels), 3))
print("\nCluster sizes:")
display(automobile_clustered["Cluster"].value_counts().sort_index().to_frame("number_of_models"))

## 6. Profile and interpret the clusters

Cluster profiles summarize the original measurements, which makes the groups easier to explain than standardized values alone.

The labels below are descriptive interpretations of this particular sample, not formal vehicle classes.

In [ ]:
profile_cols = ["mpg", "hp", "wt", "cyl", "disp", "vs", "am", "gear", "carb", "drat", "qsec"]

cluster_profiles = (
    automobile_clustered.groupby("Cluster")[profile_cols]
    .mean()
    .round(2)
)

display(cluster_profiles)

for cluster_id, group in automobile_clustered.groupby("Cluster"):
    print(f"Cluster {cluster_id} — {len(group)} models")
    print(", ".join(group["model"].tolist()))
    print()

In [ ]:
# Heatmap of standardized cluster means, useful for comparing relative patterns.
profile_z = (cluster_profiles - X[profile_cols].mean()) / X[profile_cols].std(ddof=0)

plt.figure(figsize=(13, 5))
sns.heatmap(profile_z, annot=True, fmt=".2f", cmap="vlag", center=0)
plt.title("Cluster Profiles (standardized feature means)")
plt.xlabel("Automobile features")
plt.ylabel("Cluster")
plt.tight_layout()
plt.show()

## 7. PCA visualization

PCA projects the standardized 11-dimensional data into two dimensions for visualization. PCA does not create or validate the clusters; the cluster labels come from hierarchical clustering.

The explained-variance ratio indicates how much of the original standardized feature variance is represented by the two plotted components.

In [ ]:
pca = PCA(n_components=2)
pca_coordinates = pca.fit_transform(X_scaled)

pca_df = pd.DataFrame({
    "PC1": pca_coordinates[:, 0],
    "PC2": pca_coordinates[:, 1],
    "Cluster": cluster_labels,
    "Model": automobile["model"]
})

explained = pca.explained_variance_ratio_.sum()
print(f"Variance explained by first two PCs: {explained:.1%}")

plt.figure(figsize=(11, 8))
sns.scatterplot(
    data=pca_df, x="PC1", y="PC2",
    hue="Cluster", palette="viridis", s=110
)

for _, row in pca_df.iterrows():
    plt.annotate(row["Model"], (row["PC1"], row["PC2"]),
                 xytext=(5, 4), textcoords="offset points", fontsize=8)

plt.title("Automobile Segments Visualized with PCA")
plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)")
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)")
plt.legend(title="Cluster")
plt.tight_layout()
plt.show()

## 8. Findings and business interpretation

For the Ward, three-cluster solution on all 11 standardized numeric features, the observed average profiles are:

- **Cluster 1 (15 models):** higher average fuel economy and lower average weight/horsepower than the other groups.
- **Cluster 2 (12 models):** heavier vehicles, all with eight cylinders in this sample, and lower average fuel economy.
- **Cluster 3 (5 models):** relatively high average horsepower and lower average weight than Cluster 2.

These are descriptive findings from this dataset. They do not establish causal relationships or prove that the groups represent current market segments.

### Potential business application

An automotive product team could use a similar workflow as an initial exploratory tool to identify groups of products with similar specifications. The resulting segments could inform further research into product positioning, feature differentiation, and portfolio overlap. Actual business decisions would require a larger, more current dataset and commercial variables such as price, sales, customer preferences, and geography.

## 9. Limitations and next steps

1. **Small sample:** 32 vehicles are not enough to generalize to the broader automobile market.
2. **Feature weighting:** StandardScaler gives each feature unit variance, but this does not guarantee that every feature deserves equal business importance.
3. **Binary variables:** `vs` and `am` are binary-coded. Their treatment in Euclidean distance should be justified and tested.
4. **Cluster-count sensitivity:** Several cluster counts have similar silhouette scores. The final choice should reflect the use case and interpretability.
5. **No external validation:** There is no independent ground-truth segmentation against which to validate the groups.
6. **Historical data:** The dataset is not representative of today's vehicle market.

### Recommended extensions

- Compare nine-feature and eleven-feature models.
- Test alternative distance metrics and feature weighting.
- Compare cluster stability under resampling.
- Add price, sales, customer, and market data if the goal is commercial segmentation.
- Document why a final cluster count is selected, rather than relying on one metric.

## 10. Reproducibility

To rerun this notebook:

1. Place `mtcars.csv` in the same directory as this notebook.
2. Install dependencies if needed:

   `pip install pandas numpy matplotlib seaborn scipy scikit-learn`

3. Run all cells from top to bottom.

**Portfolio summary:** This project demonstrates data inspection, preprocessing, feature standardization, hierarchical clustering, model comparison, silhouette evaluation, cluster profiling, and PCA-based visualization using Python.